<a href="https://colab.research.google.com/github/fayshalcbp/Android-app-development-in-android-studio./blob/master/RL4_SpaceInvader_random.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== 1. 시스템 패키지 =====
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3-opengl ffmpeg xvfb > /dev/null 2>&1

# ===== 2. Python 패키지 =====
!pip install -q gymnasium[atari,accept-rom-license]
!pip install -q pygame imageio imageio-ffmpeg
!pip install -q pyvirtualdisplay
!pip install -q huggingface_hub tqdm

# ===== 3. 가상 디스플레이 시작 =====
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
from pyvirtualdisplay import Display
import numpy as np  # 수학적 계산과 배열, 행렬 연산을 위한 라이브러리인 numpy를 임포트합니다.
import gymnasium as gym  # 강화 학습 환경을 제공하는 gymnasium 라이브러리를 gym이라는 이름으로 임포트합니다.
import random  # 난수 생성을 위한 라이브러리인 random을 임포트합니다.
import imageio  # 이미지를 읽고 쓰는 작업을 위한 라이브러리인 imageio를 임포트합니다.
import os  # 운영 체제와 상호작용하기 위한 다양한 기능을 제공하는 os 모듈을 임포트합니다.
import tqdm  # 반복 작업의 진행 상황을 시각적으로 표시하기 위한 라이브러리인 tqdm을 임포트합니다.

import pickle  # 객체를 파일로 저장하거나 파일에서 객체를 불러오기 위한 pickle 라이브러리를 pickle5 버전으로 임포트하고, pickle로 이름을 지정합니다.
from tqdm.notebook import tqdm  # Jupyter 노트북에서 사용하기 적합하도록 tqdm 라이브러리의 notebook 모듈에서 tqdm을 임포트합니다.

In [ ]:
!apt-get install swig cmake ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 54 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 2s (710 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 125438 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ..

In [ ]:
%%capture
!apt install python-opengl
!apt install xvfb
!pip3 install pyvirtualdisplay

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
!pip install ale-py

In [ ]:
import gymnasium as gym
import ale_py #import 안하니까 오류남 ALE 사용하려면 import 꼭 할 것..

env = gym.make("ALE/SpaceInvaders-v5", render_mode='rgb_array')
env.reset()

img = env.render()
imageio.imwrite('img.png', img)

In [ ]:
import os
import gymnasium as gym
from PIL import Image
import imageio
import pandas as pd

def clear_directory(directory):
    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)
        if os.path.isfile(item_path):
            os.remove(item_path)
        elif os.path.isdir(item_path):
            clear_directory(item_path)
            os.rmdir(item_path)

def create_final_video(out_directory, final_video_path, fps=30):
    frame_paths = []
    for episode_dir in os.listdir(out_directory):
        episode_path = os.path.join(out_directory, episode_dir)
        if os.path.isdir(episode_path):  # Ensure it's a directory
            frames = [os.path.join(episode_path, f) for f in os.listdir(episode_path) if f.endswith('.png')]
            frame_paths.extend(sorted(frames, key=lambda x: int(x.split('_')[-1].split('.')[0])))

    with imageio.get_writer(final_video_path, fps=fps, macro_block_size=1) as writer:
        for frame_path in frame_paths:
            frame = imageio.imread(frame_path)
            writer.append_data(frame)

    for frame_path in frame_paths:
        os.remove(frame_path)

# Initialize the environment
env = gym.make("ALE/SpaceInvaders-v5", render_mode='rgb_array')
episodes = 10

out_directory = 'gameplay_frames'
os.makedirs(out_directory, exist_ok=True)
clear_directory(out_directory)  # Clear the directory at the start

# Action names for clearer output
action_names = {0: "NOOP", 1: "FIRE", 2: "RIGHT", 3: "LEFT", 4: "RIGHTFIRE", 5: "LEFTFIRE"}

# Data storage for CSV
all_episode_data = []

for episode in range(episodes):
    state, info = env.reset()
    done = False
    score = 0

    episode_dir = f'episode_{episode+1}'
    episode_path = os.path.join(out_directory, episode_dir)
    os.makedirs(episode_path, exist_ok=True)

    frame_count = 0
    while not done:
        action = env.action_space.sample()  # Choose a random action
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        score += reward

        # Resize frame before saving
        frame = Image.fromarray(next_state)
        frame = frame.resize((224, 224), Image.ANTIALIAS)  # Resize frame
        frame_path = os.path.join(episode_path, f'frame_{frame_count}.png')
        frame.save(frame_path)

        # Append data for CSV
        all_episode_data.append({
            "Episode": episode + 1,
            "Step": frame_count + 1,
            "Action": action_names[action],
            "Reward": reward,
            "Total Score": score
        })
        frame_count += 1

    print(f'Episode: {episode+1}, Score: {score}')

env.close()

# Save the data to CSV
df = pd.DataFrame(all_episode_data)
csv_path = os.path.join(out_directory, "game_data.csv")
df.to_csv(csv_path, index=False)

# Create final video
final_video_path = './training_video.mp4'
create_final_video(out_directory, final_video_path, fps=10)


AttributeError: module 'PIL.Image' has no attribute 'ANTIALIAS'

In [ ]:
from google.colab import files

files.download('./training_video.mp4')